# High-Dimensional Benchmarking

Study how the current NSGA-II implementation behaves as the number of decision variables grows. This notebook uses current `optimize(...)` parameters and the backend names you would use in production.


In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from vamos import optimize
from vamos.foundation.kernel.registry import resolve_kernel
from vamos.foundation.quality_indicators import compute_hypervolume

engines = ["numpy"]
try:
    resolve_kernel("numba")
    engines.append("numba")
except ImportError:
    pass

N_VARS = [32, 128, 512, 2048]
REF_POINT = np.array([1.1, 1.1])


In [ ]:
def run_high_dim(engine: str, n_var: int):
    if engine == "numba":
        optimize("zdt1", algorithm="nsgaii", n_var=n_var, max_evaluations=200, pop_size=100, engine=engine, seed=0)
    start = time.perf_counter()
    result = optimize("zdt1", algorithm="nsgaii", n_var=n_var, max_evaluations=10000, pop_size=100, engine=engine, seed=42)
    elapsed = time.perf_counter() - start
    return {
        "engine": engine,
        "n_var": n_var,
        "runtime_s": elapsed,
        "solutions": len(result),
        "hv": compute_hypervolume(result.F, REF_POINT),
    }

rows = [run_high_dim(engine, n_var) for engine in engines for n_var in N_VARS]
hd_df = pd.DataFrame(rows)
hd_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for engine, df_engine in hd_df.groupby("engine"):
    axes[0].plot(df_engine["n_var"], df_engine["runtime_s"], marker="o", label=engine)
    axes[1].plot(df_engine["n_var"], df_engine["hv"], marker="o", label=engine)

axes[0].set_title("Runtime vs n_var")
axes[0].set_xlabel("n_var")
axes[0].set_ylabel("seconds")
axes[1].set_title("Hypervolume vs n_var")
axes[1].set_xlabel("n_var")
axes[1].set_ylabel("HV")
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

plt.tight_layout()
plt.show()
